In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# !pip install -q groq datasets seqeval


In [ ]:
import json
import time
import re
import random
from groq import Groq
from datasets import load_dataset
from seqeval.metrics import f1_score, classification_report
from kaggle_secrets import UserSecretsClient

random.seed(42)
client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))
MODEL = "llama-3.1-8b-instant"


In [ ]:
ds = load_dataset("jjzha/sayfullina")
print("Train:", len(ds["train"]))
print("Test: ", len(ds["test"]))
print("\nSample example:")
print(ds["train"][0])


<h3>Few-shot prompting baseline</h3>

In [ ]:
def get_tokens_and_tags(example):
    tokens = example["tokens"]
    # Sayfullina uses string tags directly, label is SOFT not SKILL
    tags = example["tags_skill"]
    return tokens, tags


def spans_from_bio(tokens, tags):
    spans, current = [], []
    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if current:
                spans.append(" ".join(current))
            current = [token]
        elif tag.startswith("I-") and current:
            current.append(token)
        else:
            if current:
                spans.append(" ".join(current))
            current = []
    if current:
        spans.append(" ".join(current))
    return spans


def bio_from_spans(tokens, predicted_spans):
    tags = ["O"] * len(tokens)
    for span in predicted_spans:
        span_tokens = span.strip().split()
        n = len(span_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i : i + n] == span_tokens:
                if all(tags[i + j] == "O" for j in range(n)):
                    tags[i] = "B-SOFT"
                    for j in range(1, n):
                        tags[i + j] = "I-SOFT"
                break
    return tags


In [ ]:
def sample_few_shot(dataset, n=5):
    with_skills = [
        ex for ex in dataset["train"]
        if any(t != "O" for t in ex["tags_skill"])
    ]
    return [get_tokens_and_tags(ex) for ex in random.sample(with_skills, n)]


In [ ]:
SYSTEM_PROMPT = (
    "You are a skill-span extractor for job advertisements. "
    "Given a tokenised sentence, identify every skill span — "
    "hard skills, soft skills, tools, technologies, and competencies. "
    "Reply ONLY with a valid JSON array of strings, each string being "
    "a skill phrase copied exactly as it appears in the sentence. "
    "If there are no skills, reply with []."
)

def build_user_message(tokens, few_shot_examples):
    lines = []
    for ex_tokens, ex_tags in few_shot_examples:
        spans = spans_from_bio(ex_tokens, ex_tags)
        lines.append(f'Sentence: "{" ".join(ex_tokens)}"')
        lines.append(f"Skills: {json.dumps(spans)}")
        lines.append("")
    lines.append(f'Sentence: "{" ".join(tokens)}"')
    lines.append("Skills:")
    return "\n".join(lines)


def call_groq(tokens, few_shot_examples, retries=3):
    user_msg = build_user_message(tokens, few_shot_examples)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0,
                max_tokens=256,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r'\[.*?\]', raw, re.DOTALL)
            if match:
                return json.loads(match.group())
            return []
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"Failed after {retries} attempts: {e}")
                return []


In [ ]:
print(ds["train"].features)
print(ds["train"][0])


In [ ]:
# !pip install -q tqdm


In [ ]:
from tqdm import tqdm

MAX_EXAMPLES = 200
SLEEP_BETWEEN = 2

few_shot = sample_few_shot(ds, n=5)

test_examples = list(ds["test"])
random.shuffle(test_examples)
test_examples = test_examples[:MAX_EXAMPLES]

true_tags = []
pred_tags = []

for example in tqdm(test_examples, desc="Evaluating"):
    tokens, gold = get_tokens_and_tags(example)
    predicted_spans = call_groq(tokens, few_shot)
    predicted = bio_from_spans(tokens, predicted_spans)

    true_tags.append(gold)
    pred_tags.append(predicted)

    time.sleep(SLEEP_BETWEEN)

print("\nDone.")


In [ ]:
print("=== Sayfullina — LLaMA-3.1-8B, 5-shot ===")
print(f"Examples evaluated: {len(true_tags)}")
print()
print(classification_report(true_tags, pred_tags))
f1 = f1_score(true_tags, pred_tags)
print(f"Strict F1: {f1:.4f}")
print()
print("Paper SRICL target (Qwen2.5-14B full pipeline): 61.18 F1 on Sayfullina")


In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/acme105"))


In [ ]:
print(os.listdir("/kaggle/input/datasets/acme105/skills-en-esco"))

<h3>Phase 2 - RAG Modules</h3>

In [ ]:
# !pip install -q faiss-cpu sentence-transformers groq datasets seqeval tqdm


In [ ]:
import json
import time
import re
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from groq import Groq
from datasets import load_dataset
from seqeval.metrics import f1_score, classification_report
from sentence_transformers import SentenceTransformer
import faiss
from kaggle_secrets import UserSecretsClient

random.seed(42)
client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))
MODEL = "llama-3.1-8b-instant"
embedder = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
ds = load_dataset("jjzha/sayfullina")
print("Train:", len(ds["train"]), "Test:", len(ds["test"]))


In [ ]:
def get_tokens_and_tags(example):
    return example["tokens"], example["tags_skill"]

def spans_from_bio(tokens, tags):
    spans, current = [], []
    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if current: spans.append(" ".join(current))
            current = [token]
        elif tag.startswith("I-") and current:
            current.append(token)
        else:
            if current: spans.append(" ".join(current))
            current = []
    if current: spans.append(" ".join(current))
    return spans

def bio_from_spans(tokens, predicted_spans):
    tags = ["O"] * len(tokens)
    for span in predicted_spans:
        span_tokens = span.strip().split()
        n = len(span_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i : i + n] == span_tokens:
                if all(tags[i + j] == "O" for j in range(n)):
                    tags[i] = "B-SOFT"
                    for j in range(1, n):
                        tags[i + j] = "I-SOFT"
                break
    return tags


In [ ]:
# Only use training examples that have at least one skill span
train_with_skills = [
    ex for ex in ds["train"]
    if any(t != "O" for t in ex["tags_skill"])
]

train_sentences = [" ".join(ex["tokens"]) for ex in train_with_skills]

print(f"Embedding {len(train_sentences)} training sentences...")
rag1_embeddings = embedder.encode(train_sentences, show_progress_bar=True, batch_size=64)
rag1_embeddings = rag1_embeddings.astype("float32")
faiss.normalize_L2(rag1_embeddings)

rag1_index = faiss.IndexFlatIP(rag1_embeddings.shape[1])  # inner product = cosine after normalisation
rag1_index.add(rag1_embeddings)
print(f"RAG-1 index built: {rag1_index.ntotal} entries")


In [ ]:
ESCO_PATH = "/kaggle/input/datasets/acme105/skills-en-esco/skills_en.csv"

esco = pd.read_csv(ESCO_PATH)
print("ESCO columns:", esco.columns.tolist())
print("ESCO rows:", len(esco))


In [ ]:
def build_esco_text(row):
    parts = [str(row.get("preferredLabel", ""))]
    alt = str(row.get("altLabels", ""))
    if alt and alt != "nan":
        parts.append(alt.replace("\n", ", "))
    desc = str(row.get("description", ""))
    if desc and desc != "nan":
        parts.append(desc[:200])  # cap description length
    return " | ".join(parts)

esco_texts = [build_esco_text(row) for _, row in esco.iterrows()]
esco_labels = esco["preferredLabel"].tolist()

print(f"Embedding {len(esco_texts)} ESCO skills...")
rag2_embeddings = embedder.encode(esco_texts, show_progress_bar=True, batch_size=64)
rag2_embeddings = rag2_embeddings.astype("float32")
faiss.normalize_L2(rag2_embeddings)

rag2_index = faiss.IndexFlatIP(rag2_embeddings.shape[1])
rag2_index.add(rag2_embeddings)
print(f"RAG-2 index built: {rag2_index.ntotal} entries")


In [ ]:
def retrieve_rag1(query_tokens, k=5):
    """Retrieve k most similar annotated training sentences."""
    query = " ".join(query_tokens)
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    _, indices = rag1_index.search(q_emb, k)
    results = []
    for idx in indices[0]:
        ex = train_with_skills[idx]
        tokens, tags = get_tokens_and_tags(ex)
        results.append((tokens, tags))
    return results


def retrieve_rag2(candidate_spans, k=3):
    """Retrieve top-k ESCO skill definitions for each candidate span."""
    if not candidate_spans:
        return ""
    definitions = []
    for span in candidate_spans[:5]:  # limit to 5 spans to keep prompt short
        q_emb = embedder.encode([span]).astype("float32")
        faiss.normalize_L2(q_emb)
        _, indices = rag2_index.search(q_emb, k)
        for idx in indices[0]:
            label = esco_labels[idx]
            desc  = str(esco.iloc[idx].get("description", ""))
            if desc and desc != "nan":
                definitions.append(f"- {label}: {desc[:120]}")
    # Deduplicate
    seen = set()
    unique = []
    for d in definitions:
        if d not in seen:
            seen.add(d)
            unique.append(d)
    return "\n".join(unique[:8])  # cap total definitions injected


In [ ]:
SYSTEM_PROMPT = (
    "You are a skill-span extractor for job advertisements. "
    "Given a tokenised sentence, identify every skill span — "
    "hard skills, soft skills, tools, technologies, and competencies. "
    "Reply ONLY with a valid JSON array of strings, each string being "
    "a skill phrase copied exactly as it appears in the sentence. "
    "If there are no skills, reply with []."
)

def build_user_message(tokens, few_shot_examples, esco_context):
    lines = []
    # Few-shot examples from RAG-1
    for ex_tokens, ex_tags in few_shot_examples:
        spans = spans_from_bio(ex_tokens, ex_tags)
        lines.append(f'Sentence: "{" ".join(ex_tokens)}"')
        lines.append(f"Skills: {json.dumps(spans)}")
        lines.append("")
    # ESCO grounding from RAG-2
    if esco_context:
        lines.append("Relevant skill definitions from ESCO taxonomy:")
        lines.append(esco_context)
        lines.append("")
    lines.append(f'Sentence: "{" ".join(tokens)}"')
    lines.append("Skills:")
    return "\n".join(lines)


def call_groq(tokens, few_shot_examples, esco_context, retries=3):
    user_msg = build_user_message(tokens, few_shot_examples, esco_context)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0,
                max_tokens=256,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r'\[.*?\]', raw, re.DOTALL)
            if match:
                return json.loads(match.group())
            return []
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"Failed: {e}")
                return []


In [ ]:
MAX_EXAMPLES  = 200
SLEEP_BETWEEN = 2

test_examples = list(ds["test"])
random.shuffle(test_examples)
test_examples = test_examples[:MAX_EXAMPLES]

true_tags = []
pred_tags = []

for example in tqdm(test_examples, desc="Evaluating"):
    tokens, gold = get_tokens_and_tags(example)

    # RAG-1: retrieve similar annotated sentences
    few_shot = retrieve_rag1(tokens, k=5)

    # RAG-2: first pass — use raw tokens as candidate spans for ESCO lookup
    candidate_spans = [" ".join(tokens)]
    esco_context = retrieve_rag2(candidate_spans, k=3)

    predicted_spans = call_groq(tokens, few_shot, esco_context)
    predicted = bio_from_spans(tokens, predicted_spans)

    true_tags.append(gold)
    pred_tags.append(predicted)

    time.sleep(SLEEP_BETWEEN)

print("\nDone.")


In [ ]:
print("=== Sayfullina — LLaMA-3.1-8B + RAG-1 + RAG-2, 5-shot ===")
print(f"Examples evaluated: {len(true_tags)}")
print()
print(classification_report(true_tags, pred_tags))
f1 = f1_score(true_tags, pred_tags)
print(f"Strict F1: {f1:.4f}")
print()
print("Phase 1 baseline (random 5-shot, no RAG): 0.1432 F1")
print("Paper SRICL target (Qwen2.5-14B full pipeline): 61.18 F1")


<h3>Phase 3 - Verification Step</h3>

In [ ]:
def verify_bio(tags):
    """
    Returns (is_valid, error_message).
    Checks two BIO legality rules:
      1. I-tag must follow B or I of the same type, never O
      2. Tag types must be consistent within a span
    """
    prev_tag = "O"
    for i, tag in enumerate(tags):
        if tag == "O":
            prev_tag = "O"
            continue
        prefix, label = tag.split("-", 1)  # e.g. "B", "SOFT"
        if prefix == "I":
            if prev_tag == "O":
                return False, f"I-tag at position {i} ({tag}) follows O — illegal."
            prev_prefix, prev_label = prev_tag.split("-", 1)
            if prev_label != label:
                return False, f"I-tag at position {i} ({tag}) follows {prev_tag} — type mismatch."
        prev_tag = tag
    return True, ""


def fix_bio(tags):
    """
    Best-effort repair of illegal BIO sequences:
      - I-after-O → convert to B
      - I-after-different-type → convert to B
    """
    fixed = list(tags)
    prev_tag = "O"
    for i, tag in enumerate(fixed):
        if tag == "O":
            prev_tag = "O"
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "I":
            if prev_tag == "O":
                fixed[i] = f"B-{label}"
            else:
                _, prev_label = prev_tag.split("-", 1)
                if prev_label != label:
                    fixed[i] = f"B-{label}"
        prev_tag = fixed[i]
    return fixed


In [ ]:
def call_groq_with_verify(tokens, few_shot_examples, esco_context, max_retries=3):
    user_msg = build_user_message(tokens, few_shot_examples, esco_context)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0,
                max_tokens=256,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r'\[.*?\]', raw, re.DOTALL)
            spans = json.loads(match.group()) if match else []
            bio = bio_from_spans(tokens, spans)

            valid, error = verify_bio(bio)
            if valid:
                return bio, attempt + 1  # return tags + attempts used

            # Feed the error back and retry
            messages.append({"role": "assistant", "content": raw})
            messages.append({
                "role": "user",
                "content": (
                    f"Your output produced an invalid BIO sequence: {error} "
                    f"Please reply again with a corrected JSON array of skill spans."
                )
            })
            time.sleep(1)

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"Failed: {e}")
                return ["O"] * len(tokens), max_retries

    # After all retries exhausted — apply best-effort fix
    return fix_bio(bio), max_retries


In [ ]:
MAX_EXAMPLES  = 200
SLEEP_BETWEEN = 2

test_examples = list(ds["test"])
random.shuffle(test_examples)
test_examples = test_examples[:MAX_EXAMPLES]

true_tags  = []
pred_tags  = []
total_retries = 0

for example in tqdm(test_examples, desc="Evaluating"):
    tokens, gold = get_tokens_and_tags(example)

    few_shot     = retrieve_rag1(tokens, k=5)
    esco_context = retrieve_rag2([" ".join(tokens)], k=3)

    bio, attempts = call_groq_with_verify(tokens, few_shot, esco_context, max_retries=3)
    total_retries += (attempts - 1)

    true_tags.append(gold)
    pred_tags.append(bio)

    time.sleep(SLEEP_BETWEEN)

print(f"\nDone. Total retries used: {total_retries}/{MAX_EXAMPLES}")


In [ ]:
print("=== Sayfullina — LLaMA-3.1-8B + RAG-1 + RAG-2 + Verifier ===")
print(f"Examples evaluated: {len(true_tags)}")
print()
print(classification_report(true_tags, pred_tags))
f1 = f1_score(true_tags, pred_tags)
print(f"Strict F1: {f1:.4f}")
print()
print("Phase 1 (random 5-shot, no RAG):       0.1432 F1")
print("Phase 2 (RAG-1 + RAG-2, no verifier):  0.3154 F1")
print("Paper SRICL target (Qwen2.5-14B full): 61.18 F1")
